# Capstone: Refresh Opportunity Scoring

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hassanraza04/flyrank_intern_content/blob/main/work/notebooks/capstone.ipynb)

This notebook builds a public-safe ranking for content items that deserve human review first. One row is one anonymised client-content item at a decision date. The output is a private ranked review queue, not an automated publishing decision.

## Data and validation design

The feature frame uses `fact_content_daily_performance` through DuckDB. Inputs are seven signals from two completed 28-day windows. The proxy label is 1 when next-window impressions are below 80% of current-window impressions. IDs, URLs, raw queries, GA4 fields, and all future-window values are excluded from model inputs. December 2025 through April 2026 train the candidates, May selects the method, and June is a sealed outcome cohort.

In [2]:
%pip -q install duckdb pandas pyarrow scikit-learn

import os
from pathlib import Path

if not Path('work/scripts/capstone_data.py').exists():
    !git clone -q https://github.com/hassanraza04/flyrank_intern_content.git
    %cd flyrank_intern_content

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except ImportError:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    raise RuntimeError('Set HF_TOKEN in Colab Secrets. Do not paste a token into this notebook.')

In [3]:
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

from work.scripts.capstone_data import build_feature_frame, create_connection
from work.scripts.capstone_utils import FEATURE_COLUMNS, add_baseline_score, evaluate_ranking, validate_feature_columns

cache = Path('work/outputs/capstone_features.parquet')
features = pd.read_parquet(cache) if cache.exists() else build_feature_frame(create_connection(HF_TOKEN), cache)
validate_feature_columns(FEATURE_COLUMNS)

train = features[features.cohort_id.isin(['2025-12', '2026-01', '2026-02', '2026-03', '2026-04'])]
validation = features[features.cohort_id == '2026-05']
sealed = features[features.cohort_id == '2026-06-sealed']
print({'train_rows': len(train), 'validation_rows': len(validation), 'sealed_rows': len(sealed)})

{'train_rows': 162679, 'validation_rows': 50735, 'sealed_rows': 46898}


In [4]:
models = {
    'logistic_regression': make_pipeline(SimpleImputer(strategy='median'), LogisticRegression(max_iter=1000, C=0.5, random_state=42)),
    'hist_gradient_boosting': make_pipeline(SimpleImputer(strategy='median'), HistGradientBoostingClassifier(max_depth=3, learning_rate=0.08, max_iter=150, random_state=42)),
}

rows = []
for name, model in models.items():
    model.fit(train[FEATURE_COLUMNS], train.is_declining_proxy)
    rows.append({'method': name, **evaluate_ranking(validation.is_declining_proxy, model.predict_proba(validation[FEATURE_COLUMNS])[:, 1], 100)})

validation_baseline = add_baseline_score(validation)
rows.append({'method': 'momentum_baseline', **evaluate_ranking(validation_baseline.is_declining_proxy, validation_baseline.baseline_score, 100)})
comparison = pd.DataFrame(rows).sort_values('precision_at_k', ascending=False).reset_index(drop=True)
comparison

                   method  base_rate  precision_at_k   roc_auc
0  hist_gradient_boosting   0.446575            0.79  0.593460
1       momentum_baseline   0.446575            0.69  0.521684
2     logistic_regression   0.446575            0.47  0.521664

## Sealed June result

Gradient boosting is frozen after May selection, retrained on development plus validation rows, and evaluated against the same baseline on June. This is an observed ranking result. It does not prove that refreshing a page changes Google's rankings or future visibility.

In [5]:
selected_model = models['hist_gradient_boosting'].fit(pd.concat([train, validation])[FEATURE_COLUMNS], pd.concat([train, validation]).is_declining_proxy)
sealed_baseline = add_baseline_score(sealed)
baseline_metrics = evaluate_ranking(sealed.is_declining_proxy, sealed_baseline.baseline_score, 100)
model_scores = selected_model.predict_proba(sealed[FEATURE_COLUMNS])[:, 1]
model_metrics = evaluate_ranking(sealed.is_declining_proxy, model_scores, 100)
print({'selected_method': 'hist_gradient_boosting', 'sealed_precision_at_100': model_metrics['precision_at_k'], 'baseline_precision_at_100': baseline_metrics['precision_at_k'], 'sealed_model_roc_auc': model_metrics['roc_auc'], 'baseline_roc_auc': baseline_metrics['roc_auc'], 'sealed_base_rate': model_metrics['base_rate']})

{'selected_method': 'hist_gradient_boosting', 'sealed_precision_at_100': 0.65, 'baseline_precision_at_100': 0.38, 'sealed_model_roc_auc': 0.56011441057918, 'baseline_roc_auc': 0.5044254995250972, 'sealed_base_rate': 0.5189347093692694}


## Action and limitations

Use the top 100 private scores as the start of a human review queue. Review meaningful visible items with falling recent momentum, inspect search intent when position worsens, inspect snippets only after confirming relevance and visibility, and monitor weak signals. The model is useful for prioritising limited editorial attention. It is not an automatic action engine and it does not make causal claims.

Built on the [FlyRank ML Internship dataset](https://flyrank.ai).